# GameTheory-04h — Jeux de routage / congestion et prix de l’anarchie

**Navigation** : [Version Python](GameTheory-04h-Routing-PriceOfAnarchy-Python.ipynb) | [Index GameTheory](README.md)

**Compagnon Lean** : [Formalisation GameTheory-04h](GameTheory-04h-Routing-PriceOfAnarchy-Lean.ipynb)

**Objectifs** : calculer un équilibre de Wardrop et un optimum social sur le réseau de Pigou, comparer leurs coûts, interpréter le PoA et vérifier les cas intérieurs et frontières.

**Prérequis** : fonctions affines, dérivée d’une fonction quadratique et équilibre de Nash.

**Durée estimée** : 35 minutes.

Ce carnet implémente les calculs analytiques en C# avec la bibliothèque standard .NET 9. Il complète la version Python et conserve le compagnon Lean, qui porte la formalisation.

**Références** : Tim Roughgarden, [*Selfish Routing and the Price of Anarchy*](https://mitpress.mit.edu/9780262182430/selfish-routing-and-the-price-of-anarchy/), MIT Press, 2005 ; Tim Roughgarden et Éva Tardos, [“How Bad is Selfish Routing?”](https://www.timroughgarden.org/papers/routing.pdf), *Journal of the ACM* 49(2), 236–259, 2002 (FOCS 2000).


In [1]:
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Xml.Linq;
CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
(double XA,double XB,double Cost) PigouOptimum(double a,double d) {
    CheckDomain(a,d);
    double t=a*d;
    double x=Math.Min(d,0.5/a), y=d-x;
    double normalizedCost=t<=0.5?t:1.0-0.25/t;
    return (x,y,d*normalizedCost);
}
(double XA,double XB,double Cost) PigouWardrop(double a,double d) {
    CheckDomain(a,d);
    double t=a*d;
    double x=t<=1.0?d:1.0/a, y=d-x;
    double normalizedCost=t<=1.0?t:1.0;
    return (x,y,d*normalizedCost);
}
double PigouPoa(double a,double d) {
    CheckDomain(a,d);
    double t=a*d;
    return t<=0.5?1.0:Math.Min(t,1.0)/(1.0-0.25/t);
}
void CheckDomain(double a,double d) {
    if(!double.IsFinite(a)||!double.IsFinite(d)||a<=0||d<=0)
        throw new ArgumentOutOfRangeException("alpha et D doivent etre finis et strictement positifs");
}
Console.WriteLine("Runtime .NET " + Environment.Version);
Console.WriteLine("Fonctions analytiques chargees : Wardrop, optimum borne et PoA normalisé ; validation de domaine active.");
var fixtures = new[]{(1.0,0.1,1.0),(0.25,1.0,1.0),(0.5,1.0,1.0),(1.0,1.0,4.0/3),(2.0,2.0,16.0/15),(1e-300,1e-300,1.0),(1e-300,1e300,4.0/3),(1e300,1e300,1.0)};
bool gridOk=true, wardropOk=true, poaFinite=true;
foreach(var f in fixtures){
    var o=PigouOptimum(f.Item1,f.Item2); var u=PigouWardrop(f.Item1,f.Item2);
    gridOk &= o.XA>=0 && o.XA<=f.Item2 && o.XB>=0 && double.IsFinite(o.Cost) && Math.Abs(o.XA+o.XB-f.Item2)<1e-12*Math.Max(1,f.Item2) && Math.Abs(PigouPoa(f.Item1,f.Item2)-f.Item3)<1e-10;
    wardropOk &= double.IsFinite(u.XA)&&double.IsFinite(u.XB)&&u.XA>=0&&u.XB>=0&&Math.Abs(u.XA+u.XB-f.Item2)<1e-12*Math.Max(1,f.Item2)&&(u.XB==0||Math.Abs(f.Item1*u.XA-1)<1e-10);
    poaFinite &= double.IsFinite(PigouPoa(f.Item1,f.Item2));
    if(f.Item1<1e100&&f.Item2<1e100) for(int i=0;i<=10000;i++){double x=f.Item2*i/10000.0; double cost=f.Item1*x*x+(f.Item2-x); if(o.Cost>cost+1e-8) gridOk=false;}
}
bool invalidRejected=false; try{PigouPoa(0,1);}catch(ArgumentOutOfRangeException){invalidRejected=true;}
Console.WriteLine($"Fixtures normales + extrêmes : {fixtures.Length}; optimum/oracle grille (cas bornés)={gridOk}; Wardrop/conservation={wardropOk}; PoA toujours fini={poaFinite}; alpha=0 rejeté={invalidRejected}");

Runtime .NET 10.0.14


Fonctions analytiques chargees : Wardrop, optimum borne et PoA normalisé ; validation de domaine active.


Fixtures normales + extrêmes : 8; optimum/oracle grille (cas bornés)=True; Wardrop/conservation=True; PoA toujours fini=True; alpha=0 rejeté=True


## 1. L'instance canonique : l'exemple de Pigou

Deux chemins en parallèle relient la source `s` à la destination `t`, avec une demande totale `D` d'usagers :

- **Autoroute** (lien congestionné) : le coût par usager est proportionnel au nombre d'usagers dessus, `c_A(x_A) = alpha * x_A`.
- **Route locale** (lien libre) : coût constant `c_B = 1`, sans congestion.

Chaque usager choisit un chemin. C'est l'exemple fondateur qui donne son nom au phénomène : la route locale à coût constant égal à 1 conduit les usagers égoïstes à se jeter tous sur l'autoroute, alors que l'optimum social les répartirait.

In [2]:
double alpha = 1.0, D = 1.0;
var opt = PigouOptimum(alpha, D);
Console.WriteLine($"Optimum social : x_A={opt.XA:F3}, x_B={opt.XB:F3}, cout={opt.Cost:F4}");

Optimum social : x_A=0.500, x_B=0.500, cout=0.7500


## 2. L'équilibre de Wardrop (équilibre d'utilisateur / Nash)

À l'équilibre de Wardrop, **aucun usager ne peut réduire son coût personnel en changeant unilatéralement de chemin**. Tant que le coût de l'autoroute `c_A(x_A) = alpha * x_A` est inférieur au coût de la route locale (`1`), les usagers préfèrent l'autoroute : ils s'y massent jusqu'à ce que `alpha * x_A` atteigne `1`. Au-delà, plus personne n'y gagne à s'y ajouter.

Cet équilibre est un **équilibre de Nash** du jeu de routage : la stratégie de chaque joueur (son chemin) est une meilleure réponse à celle des autres.

In [3]:
var ue = PigouWardrop(alpha, D);
Console.WriteLine($"Equilibre de Wardrop : x_A={ue.XA:F3}, x_B={ue.XB:F3}, cout={ue.Cost:F4}");
Console.WriteLine($"Conservation : {ue.XA+ue.XB:F3} = D={D:F3}; chemins utilises : {(ue.XB == 0 ? "A seul" : "A et B")}");

Equilibre de Wardrop : x_A=1.000, x_B=0.000, cout=1.0000


Conservation : 1.000 = D=1.000; chemins utilises : A seul


### Lecture de l’équilibre mesuré

La sortie donne `x_A=1`, `x_B=0` et un coût social total de 1. Aucun usager du chemin A ne peut gagner en basculant seul vers le chemin de coût 1 ; l’exercice suivant généralise ces conditions.


<a id="exercice-1"></a>

### Exercice 1 — Équilibre de Wardrop général

Pour une instance à **2 liens en parallèle** (autoroute `c(x) = alpha*x`, route libre `c = 1`) et une demande `D`, calcule l'équilibre de Wardrop, c'est-à-dire la répartition `(x_A, x_B)` des usagers à l'équilibre.

**Indice** : à l'équilibre, un usager ne gagne rien à changer de lien — compare le coût `c_A(x_A)` de l'autoroute au coût `1` de la route locale.

In [4]:
(double XA,double XB)? WardropEtudiant(double a,double demande) { return null; } // TODO etudiant
(double XA,double XB) WardropOracle(double a,double d)=>(Math.Min(d,1.0/a),d-Math.Min(d,1.0/a));
var wardropFixtures=new[]{(A:1.0,D:0.4),(A:0.5,D:2.0),(A:2.0,D:1.0),(A:0.25,D:1.0)};
bool AcceptWardrop((double XA,double XB)? actual,double a,double d){
    if(actual is not { } answer)return false;
    var expected=WardropOracle(a,d);
    bool pathConditions=answer.XB==0? a*answer.XA<=1.0+1e-10 : Math.Abs(a*answer.XA-1.0)<1e-10;
    return double.IsFinite(answer.XA)&&double.IsFinite(answer.XB)&&answer.XA>=0&&answer.XB>=0&&Math.Abs(answer.XA+answer.XB-d)<1e-10&&Math.Abs(answer.XA-expected.XA)<1e-10&&Math.Abs(answer.XB-expected.XB)<1e-10&&pathConditions;
}
bool wardropAnswerPresent=WardropEtudiant(1,1) is not null;
bool wardropFixturesPass=wardropFixtures.All(f=>AcceptWardrop(WardropEtudiant(f.A,f.D),f.A,f.D));
bool wardropMutantsRejected=wardropFixtures.All(f=>!AcceptWardrop((0,0),f.A,f.D)&&!AcceptWardrop((double.NaN,double.NaN),f.A,f.D));
string wardropStatus = !wardropAnswerPresent ? "à compléter" : wardropFixturesPass ? "réponse validée sur les fixtures" : "réponse incorrecte sur les fixtures";
Console.WriteLine($"Exercice 1 : {wardropStatus}; fixtures={wardropFixtures.Length}; mutants constants/NaN rejetés={wardropMutantsRejected}.");

Exercice 1 : à compléter; fixtures=4; mutants constants/NaN rejetés=True.


## 3. Le Price of Anarchy — mesuré

Le **price of anarchy** rapporte le coût à l'équilibre égoïste au coût à l'optimum social :

$$\text{PoA} = \frac{\text{cout total à l'équilibre de Wardrop}}{\text{cout total à l'optimum social}}$$

C'est le chiffre central du carnet : il **quantifie** le prix de l'intérêt individuel décentralisé. PoA = 1 signifie que l'équilibre est déjà optimal ; plus il est grand, plus l'égoïsme collective est coûteux.

In [5]:
double poaCanonical = ue.Cost / opt.Cost;
Console.WriteLine($"Price of Anarchy = {ue.Cost:F4} / {opt.Cost:F4} = {poaCanonical:F4}");
Console.WriteLine($"Surcoût social total mesuré : {100*(poaCanonical-1):F1} % ; controle 4/3 : {Math.Abs(poaCanonical-4.0/3)<1e-12}");

Price of Anarchy = 1.0000 / 0.7500 = 1.3333


Surcoût social total mesuré : 33.3 % ; controle 4/3 : True


### Lecture du ratio mesuré

La sortie compare les coûts sociaux totaux : 1 à Wardrop contre 0,75 à l’optimum, soit un PoA de 4/3 et un surcoût social total de 33,3 %.


<a id="exercice-2"></a>

### Exercice 2 — La fonction PoA

Écris une fonction `poa(alpha, D)` qui renvoie le **price of anarchy mesuré** de l'instance de Pigou généralisée (autoroute `alpha*x`, route libre `1`, demande `D`). Elle doit reproduire la valeur `4/3` pour `alpha = 1, D = 1`.

In [6]:
double? PoaEtudiant(double a,double demande) { return null; } // TODO etudiant
var poaFixtures=new[]{(A:1.0,D:1.0,Expected:4.0/3),(A:1.0,D:0.1,Expected:1.0),(A:0.25,D:1.0,Expected:1.0),(A:2.0,D:2.0,Expected:16.0/15),(A:1e-300,D:1e-300,Expected:1.0),(A:1e-300,D:1e300,Expected:4.0/3),(A:1e300,D:1e300,Expected:1.0)};
bool AcceptPoa(double? actual,double expected)=>actual is double value&&double.IsFinite(value)&&Math.Abs(value-expected)<1e-9;
bool poaAnswerPresent=PoaEtudiant(1,1) is not null;
bool poaFixturesPass=poaFixtures.All(f=>AcceptPoa(PoaEtudiant(f.A,f.D),f.Expected));
bool poaMutantsRejected=poaFixtures.All(f=>!AcceptPoa(0.0,f.Expected)&&!AcceptPoa(double.NaN,f.Expected));
string poaStatus = !poaAnswerPresent ? "à compléter" : poaFixturesPass ? "réponse validée sur les fixtures" : "réponse incorrecte sur les fixtures";
Console.WriteLine($"Exercice 2 : {poaStatus}; cas canoniques/frontières/extrêmes={poaFixtures.Length}; mutants constants/NaN rejetés={poaMutantsRejected}.");

Exercice 2 : à compléter; cas canoniques/frontières/extrêmes=7; mutants constants/NaN rejetés=True.


## 4. La borne 4/3 (Roughgarden–Tardos, 2002)

**Théorème.** Pour un réseau à **une seule commodity** dont les coûts de lien sont **affines non décroissantes** `c_e(x) = a_e x_e + b_e` avec `a_e >= 0` et `b_e >= 0` (modèle à flux fractionnaire), le price of anarchy est **borné par 4/3** :

$$\text{PoA} \le 4/3$$

Cette borne est **serrée** : l'instance de Pigou mesurée ci-dessus la **atteint exactement** en `α = 1` (PoA = 4/3) — elle en est le **cas d'égalité**. C'est le chiffre central de la branche : l'écart maximal entre l'équilibre égoïste et l'optimum social, pour toute congestion linéaire, est de **33 %**. (Le « 5/2 » souvent cité relève des jeux de congestion **atomiques** — joueurs indivisibles — de Christodoulou–Koutsoupias, un modèle distinct de celui de ce carnet.)

> **Portée du compagnon Lean** : il formalise un résultat atomique distinct : un équilibre minimisant le potentiel de Rosenthal a un coût au plus deux fois l’optimum, et ses fixtures montrent aussi un équilibre de Nash coûtant `15/7` fois l’optimum. Il ne formalise pas ici la borne non atomique 4/3.

## 5. Sensibilité : le PoA en fonction de la pente de congestion

Faisons varier la pente `alpha` de l'autoroute (à demande `D = 1` fixe) et **mesurons** le PoA. On attend : plus la congestion est raide, plus l'écart se referme — le PoA **décroît** vers 1. Le maximum tombe en `alpha = 1`, où Pigou réalise l'égalité de la borne ; sur tout ce domaine le PoA mesuré doit rester **≤ 4/3**.

In [7]:
double[] alphas = {1.0,1.2,1.4,1.5,1.8,2.0};
var sensitivity = alphas.Select(a => (Alpha:a, Poa:PigouPoa(a,1.0))).ToArray();
Console.WriteLine("alpha | PoA | <= 4/3");
foreach(var row in sensitivity) Console.WriteLine($"{row.Alpha:F1} | {row.Poa:F4} | {row.Poa <= 4.0/3+1e-12}");

alpha | PoA | <= 4/3


1.0 | 1.3333 | True


1.2 | 1.2632 | True


1.4 | 1.2174 | True


1.5 | 1.2000 | True


1.8 | 1.1613 | True


2.0 | 1.1429 | True


### Lecture des mesures

Pour `alpha` égal à 1, 1,2, 1,4, 1,5, 1,8 et 2, les valeurs observées sont respectivement 1,3333, 1,2632, 1,2174, 1,2000, 1,1613 et 1,1429. Dans cette famille à demande fixe, le coût de Wardrop vaut 1 à partir de `alpha = 1`, tandis que l’optimum social s’en approche ; le PoA mesuré décroît donc vers 1. La borne 4/3 s’applique au modèle affine à flux fractionnaire décrit dans l’énoncé, et l’instance canonique l’atteint.


### Transition vers l’exercice de sensibilité

L’exercice suivant reprend ces calculs en faisant varier la demande plutôt que la pente.


<a id="exercice-3"></a>

### Exercice 3 — Le PoA en fonction de la demande

Pour l'instance canonique (`alpha = 1`), calcule le PoA pour une **série de valeurs de la demande** `D` sur `[1, 4]` et décris la tendance. Que devient le PoA quand la demande croît ? (Rappel : à l'équilibre, tous sont sur l'autoroute tant que `alpha * D <= 1` ; au-delà, le flux se répartit.)

In [8]:
double? PoaPourDemandeEtudiant(double demande,double a=1.0) { return null; } // TODO etudiant
double[] demandes={1,2,3,4};
double OraclePoa(double a,double d){double t=a*d;return t<=0.5?1.0:Math.Min(t,1.0)/(1.0-0.25/t);}
var poaDemandResults=demandes.Select(d=>PoaPourDemandeEtudiant(d,1.0)).ToArray();
bool AcceptDemand(double? actual,double expected)=>actual is double value&&double.IsFinite(value)&&Math.Abs(value-expected)<1e-9;
bool demandSeriesPass=poaDemandResults.Length==demandes.Length&&poaDemandResults.Select((v,i)=>AcceptDemand(v,OraclePoa(1.0,demandes[i]))).All(ok=>ok);
bool demandMutantsRejected=demandes.All(d=>!AcceptDemand(0.5,OraclePoa(1.0,d))&&!AcceptDemand(double.NaN,OraclePoa(1.0,d)));
bool demandAnswerPresent=PoaPourDemandeEtudiant(1) is not null;
string demandStatus = !demandAnswerPresent ? "à compléter" : demandSeriesPass ? "réponse validée sur la série" : "réponse incorrecte sur la série";
Console.WriteLine($"Exercice 3 : {demandStatus}; demandes=[{string.Join(", ",demandes)}]; dimension attendue={demandes.Length}; mutants constants/NaN rejetés={demandMutantsRejected}.");

Exercice 3 : à compléter; demandes=[1, 2, 3, 4]; dimension attendue=4; mutants constants/NaN rejetés=True.


## 6. Récapitulatif des exercices

Les trois fonctions restent des stubs à compléter :

1. [Équilibre de Wardrop général](#exercice-1--équilibre-de-wardrop-général) — conservation et coûts des chemins utilisés.
2. [Fonction PoA](#exercice-2--la-fonction-poa) — cas intérieur, frontières et domaines numériques extrêmes.
3. [PoA selon la demande](#exercice-3--le-poa-en-fonction-de-la-demande) — calculer `D=1,2,3,4` et décrire les valeurs obtenues.

Les cellules de contrôle appellent les réponses réelles et confrontent les sorties aux oracles ; elles n’apportent pas les solutions aux exercices.


## 7. À retenir

Sur l’instance de Pigou `alpha = D = 1`, l’équilibre de Wardrop envoie toute la demande sur le lien congestionné : le coût total vaut 1. L’optimum social partage le flux en deux parts égales et coûte 0,75 ; le rapport vaut donc 4/3. L’optimum doit respecter la contrainte de flux `0 <= x_A <= D` : si le point critique intérieur sort de cet intervalle, le minimum est sur la frontière.

La borne de Roughgarden–Tardos concerne les réseaux à flux fractionnaire et coûts affines ; le 5/2 appartient à un modèle atomique distinct. Le compagnon Lean vérifie un résultat atomique distinct : facteur 2 pour un équilibre minimisant le potentiel, et un témoin à 15/7 pour un autre équilibre de Nash. Il ne formalise pas la borne non atomique 4/3.

## Vérifiez votre compréhension

<details><summary>Pourquoi l’optimum doit-il être borné par D ?</summary>

Le flux sur un lien ne peut dépasser la demande totale. Si `1/(2 alpha) > D`, la fonction de coût social est encore décroissante sur le domaine faisable et son minimum est atteint en `x_A = D`.

</details>

<details><summary>Pourquoi le PoA canonique vaut-il 4/3 ?</summary>

Le coût d’équilibre vaut 1 et le coût optimal vaut 0,75 ; leur quotient vaut `1 / 0,75 = 4/3`.

</details>

## Pour aller plus loin

- [Version Python du calcul](GameTheory-04h-Routing-PriceOfAnarchy-Python.ipynb)
- [Compagnon Lean](GameTheory-04h-Routing-PriceOfAnarchy-Lean.ipynb)
- Tim Roughgarden et Éva Tardos, [“How Bad is Selfish Routing?”](https://www.timroughgarden.org/papers/routing.pdf), *Journal of the ACM* 49(2), 236–259, 2002 (FOCS 2000).
- Tim Roughgarden, [*Selfish Routing and the Price of Anarchy*](https://mitpress.mit.edu/9780262182430/selfish-routing-and-the-price-of-anarchy/), MIT Press, 2005.
